In [128]:
from xgboost import XGBClassifier, XGBModel
import xgboost as xg
from sklearn.metrics import accuracy_score, mean_squared_error, roc_auc_score 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd

In [2]:
loan = pd.read_csv(os.path.abspath(os.path.join(os.getcwd() + '\\..\\data\\train_loanpred.csv')))
tree_loan = loan.loc[:, ~loan.columns.isin(["Loan_ID"])]
print(tree_loan.shape)

(614, 12)


In [83]:
tree_loan["Loan_Status"].value_counts()

Loan_Status
Y    422
N    192
Name: count, dtype: int64

In [65]:
loan_test = pd.read_csv(os.path.abspath(os.path.join(os.getcwd() + '\\..\\data\\test_loanpred.csv')))
x_test = loan_test.loc[:, ~loan_test.columns.isin(["Loan_ID"])]
print(x_test.shape)

(367, 11)


In [4]:
tree_loan.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,0,NaN,360.0,1.0,Urban,Y
1,Male,Yes,1,Graduate,No,4583,1508,128.0,360.0,1.0,Rural,N
2,Male,Yes,0,Graduate,Yes,3000,0,66.0,360.0,1.0,Urban,Y
3,Male,Yes,0,Not Graduate,No,2583,2358,120.0,360.0,1.0,Urban,Y
4,Male,No,0,Graduate,No,6000,0,141.0,360.0,1.0,Urban,Y


In [3]:
tree_loan.isnull().sum().sort_values(ascending=False)

Credit_History       50
Self_Employed        32
LoanAmount           22
Dependents           15
Loan_Amount_Term     14
Gender               13
Married               3
Education             0
CoapplicantIncome     0
ApplicantIncome       0
Property_Area         0
Loan_Status           0
dtype: int64

In [8]:
X = tree_loan.iloc[:, :-1]
Y = tree_loan.iloc[:, -1]
x_train, x_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=22)

In [28]:
cat_cols = ["Property_Area", "Dependents", "Gender", "Married", "Education", "Self_Employed"]


In [69]:
for col in cat_cols: 
    x_train[col] = x_train[col].astype('category')
    x_val[col] =  x_val[col].astype('category')
    x_test[col] = x_test[col].astype('category')

C:\Users\nedwi\AppData\Local\Temp\ipykernel_19732\1117333741.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_test[col] = x_test[col].astype('category')
C:\Users\nedwi\AppData\Local\Temp\ipykernel_19732\1117333741.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_test[col] = x_test[col].astype('category')
C:\Users\nedwi\AppData\Local\Temp\ipykernel_19732\1117333741.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexe

In [71]:
x_train.dtypes

Gender               category
Married              category
Dependents           category
Education            category
Self_Employed        category
ApplicantIncome         int64
CoapplicantIncome       int64
LoanAmount            float64
Loan_Amount_Term      float64
Credit_History        float64
Property_Area        category
dtype: object

In [33]:
y_train_mapped = y_train.map(lambda x: 1 if x == 'Y' else 0)
y_val_mapped = y_val.map(lambda x: 1 if x == 'Y' else 0)

In [32]:
model = XGBClassifier(enable_categorical=True) 
model.fit(x_train, y_train_mapped)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [35]:
pred = model.predict(x_val)
accuracy_score(y_val_mapped, pred)

0.7154471544715447

In [82]:
pos_val = y_train_mapped[y_train_mapped == 0].shape[0]/ y_train_mapped[y_train_mapped == 1].shape[0]
pos_val

0.4526627218934911

In [159]:
xgb = XGBClassifier(n_estimators=500, 
                    base_score=0.5,
                    early_stopping_rounds=20, 
                    learning_rate = 0.1,
                    subsample = 0.8,
                    max_depth=3,
                    tree_method = 'auto', #hist
                    max_bin = 256,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    gamma=0.1,
                    reg_alpha = 0,
                    reg_lambda = 1,
                    enable_categorical=True,
                    missing=np.nan,
                    #scale_pos_weight=pos_val,
                    random_state=42
                    )

xgb.fit(x_train, y_train_mapped, eval_set=[(x_train, y_train_mapped), (x_val,y_val_mapped)], verbose=True)
#dont use testing data in eval

[0]	validation_0-logloss:0.65479	validation_1-logloss:0.65685
[1]	validation_0-logloss:0.62304	validation_1-logloss:0.62757
[2]	validation_0-logloss:0.59601	validation_1-logloss:0.60237
[3]	validation_0-logloss:0.57300	validation_1-logloss:0.58103
[4]	validation_0-logloss:0.55411	validation_1-logloss:0.56237
[5]	validation_0-logloss:0.53813	validation_1-logloss:0.54571
[6]	validation_0-logloss:0.52416	validation_1-logloss:0.53115
[7]	validation_0-logloss:0.51159	validation_1-logloss:0.52103
[8]	validation_0-logloss:0.50134	validation_1-logloss:0.51204
[9]	validation_0-logloss:0.49171	validation_1-logloss:0.50175
[10]	validation_0-logloss:0.48387	validation_1-logloss:0.49528
[11]	validation_0-logloss:0.47691	validation_1-logloss:0.48793
[12]	validation_0-logloss:0.47015	validation_1-logloss:0.48165
[13]	validation_0-logloss:0.46392	validation_1-logloss:0.47959
[14]	validation_0-logloss:0.45946	validation_1-logloss:0.47713
[15]	validation_0-logloss:0.45487	validation_1-logloss:0.47654
[1

,objective,'binary:logistic'
,base_score,0.5
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,20
,enable_categorical,True
,eval_metric,'logloss'


In [160]:
print(f'Best iteration is {xgb.best_iteration} and \
   \n trainging best score is {xgb.evals_result()["validation_0"]["logloss"][xgb.best_iteration]} \
    \n validation best score is {xgb.best_score} after early stopping')

Best iteration is 45 and    
 trainging best score is 0.3697728935151139     
 validation best score is 0.44882087167201 after early stopping


In [162]:
xgb_pred = xgb.predict(x_val)
accuracy_score(y_val_mapped, xgb_pred)

0.8048780487804879

In [153]:
import xgboost as xgb 
dtrain = xgb.DMatrix(x_train, label=y_train_mapped, enable_categorical=True) 
params = { 
                    #'n_estimators':500, 
                    'base_score':0.5,
                    #'early_stopping_rounds':20, 
                    'learning_rate': 0.1,
                    'subsample' : 0.8,
                    'max_depth':3,
                    'tree_method' : 'auto', #hist
                    'max_bin' :256,
                    'objective':"binary:logistic",
                    'eval_metric':"logloss",
                    'gamma':0.1,
                    'reg_alpha': 0,
                    'reg_lambda': 1,
                    'enable_categorical':True,
         }
cv_results = xgb.cv( params, dtrain, num_boost_round=1000, nfold=5, early_stopping_rounds=20, seed=42 )
cv_results

C:\Users\nedwi\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:225: UserWarning: [22:55:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "enable_categorical" } are not used.

  return getattr(self.bst, name)(*args, **kwargs)
C:\Users\nedwi\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:231: UserWarning: [22:55:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "enable_categorical" } are not used.

  self.bst.update(self.dtrain, iteration, fobj)


,train-logloss-mean,train-logloss-std,test-logloss-mean,test-logloss-std
0,0.653714,0.002339,0.657214,0.005023
1,0.622883,0.004325,0.630270,0.009954
2,0.597019,0.006709,0.609193,0.015009
3,0.573857,0.008034,0.588795,0.019070
4,0.554628,0.009049,0.572168,0.022841
5,0.537200,0.009935,0.558697,0.026405
6,0.523362,0.010597,0.547881,0.029770
7,0.510387,0.011149,0.538716,0.033247
8,0.498933,0.011704,0.532234,0.036683
9,0.489658,0.012502,0.526070,0.039213


In [161]:
cv_results["train-logloss-mean"].argmin()

np.int64(21)

In [154]:
best_num_boost_round = len(cv_results)
best_num_boost_round

22

In [155]:
xgb_clf = XGBClassifier( n_estimators=best_num_boost_round, **params ) 
xgb_clf.fit(x_train, y_train_mapped) 
preds = xgb_clf.predict(x_val)

In [156]:
accuracy_score(y_val_mapped, preds)

0.8292682926829268

In [75]:
pred = xgb.predict(x_test)

In [ ]:
x_cat= pd.get_dummies(x_train[cat_cols], drop_first=True)
x_val_cat = pd.get_dummies(x_val[cat_cols], drop_first=True)
x_val_cat = x_val_cat.reindex(columns=x_cat.columns, fill_value=0)
